In [1]:
# ============================================================
# 3MP Multimodal Diagnostic Platform
# Publication Figure Generator
# ============================================================

from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_curve,
    auc,
    confusion_matrix,
)


# ============================================================
# 1. Repository and directory setup
# ============================================================

REPO_NAME = "3MP-Multimodal-Diagnostic-Platform"

REPO_URL = (
    "https://github.com/GanghaoLiang/"
    "3MP-Multimodal-Diagnostic-Platform.git"
)


def find_repo_root(start_path):
    """
    Search the current directory and its parent directories
    for the 3MP repository.

    The repository root is identified by the presence of
    the Publication_Results directory.
    """
    start_path = Path(start_path).resolve()

    for candidate in [start_path, *start_path.parents]:
        if (candidate / "Publication_Results").is_dir():
            return candidate

    return None


# First check whether the notebook is already being executed
# inside a cloned copy of the repository.
repo_root = find_repo_root(Path.cwd())


# When the notebook is opened directly in Google Colab,
# the GitHub repository is not automatically mounted.
# Clone the repository if necessary.
if repo_root is None and "google.colab" in sys.modules:

    clone_dir = Path("/content") / REPO_NAME

    if not clone_dir.exists():

        print(
            "Repository not found locally. "
            "Cloning from GitHub..."
        )

        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                REPO_URL,
                str(clone_dir),
            ],
            check=True,
        )

    repo_root = find_repo_root(clone_dir)


if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the repository containing "
        "'Publication_Results'.\n"
        "Please run this notebook from within the cloned "
        "3MP-Multimodal-Diagnostic-Platform repository."
    )


# Fixed source data used for the manuscript figures.
data_dir = repo_root / "Publication_Results"

# Newly generated figures are deliberately kept separate
# from the fixed publication source data.
output_dir = repo_root / "Generated_Figures"

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


print(f"Repository root:\n{repo_root}")

print(
    f"\nReading publication source data from:\n"
    f"{data_dir}"
)

print(
    f"\nGenerated figures will be saved to:\n"
    f"{output_dir}"
)


# ============================================================
# 2. Analysis definitions
# ============================================================

class_names = [
    "Normal",
    "Scar",
    "Inflamed",
    "Tumor",
]

models_list = [
    "Bio",
    "Mech",
    "Fused",
]

model_titles = [
    "Biochemical Baseline",
    "Biomechanical Baseline",
    "Multimodal Fusion",
]

# Colors used for the three model ROC curves.
colors = [
    "#1f77b4",
    "#2ca02c",
    "#d62728",
]

# Colors used for the four tissue classes in t-SNE plots.
palette = [
    "#1f77b4",
    "#2ca02c",
    "#ff7f0e",
    "#d62728",
]


# ============================================================
# 3. Publication-style plotting parameters
# ============================================================

plt.rcParams.update(
    {
        "font.size": 10,
        "font.family": "sans-serif",
        "axes.linewidth": 1.2,
        "xtick.major.width": 1.2,
        "ytick.major.width": 1.2,

        # Store text as editable TrueType text in PDF files.
        "pdf.fonttype": 42,
    }
)


# ============================================================
# 4. Check required publication source files
# ============================================================

roc_file = (
    data_dir
    / "Ablation_ROC_Probabilities.csv"
)

tsne_files = {
    prefix: (
        data_dir
        / f"{prefix}_tSNE_Coordinates.csv"
    )
    for prefix in models_list
}


required_files = [
    roc_file,
    *tsne_files.values(),
]


missing_files = [
    file_path
    for file_path in required_files
    if not file_path.is_file()
]


if missing_files:

    missing_text = "\n".join(
        str(path)
        for path in missing_files
    )

    raise FileNotFoundError(
        "The following required publication source files "
        "were not found:\n"
        f"{missing_text}"
    )


print(
    "\nAll required publication source files were found."
)


# ============================================================
# 5. Load OOF prediction probabilities
# ============================================================

df_roc = pd.read_csv(
    roc_file
)


required_roc_columns = [
    "True_Label",
]

for prefix in models_list:
    for class_name in class_names:

        required_roc_columns.append(
            f"{prefix}_Prob_{class_name}"
        )


missing_columns = [
    column
    for column in required_roc_columns
    if column not in df_roc.columns
]


if missing_columns:

    raise ValueError(
        "Missing required columns in "
        "Ablation_ROC_Probabilities.csv:\n"
        + "\n".join(missing_columns)
    )


y_true = df_roc[
    "True_Label"
].to_numpy(
    dtype=np.int32
)


if len(y_true) != 384:

    raise ValueError(
        f"Expected 384 OOF samples, "
        f"but found {len(y_true)}."
    )


if not set(
    np.unique(y_true)
).issubset(
    {0, 1, 2, 3}
):

    raise ValueError(
        "True_Label contains unexpected class values."
    )


print(
    f"Loaded {len(y_true)} out-of-fold predictions."
)


# ============================================================
# 6. Generate three confusion matrices
# ============================================================

print(
    "\nGenerating confusion matrices..."
)


for prefix, title in zip(
    models_list,
    model_titles,
):

    probability_columns = [
        f"{prefix}_Prob_{class_name}"
        for class_name in class_names
    ]

    probs = df_roc[
        probability_columns
    ].to_numpy()


    if not np.all(
        np.isfinite(probs)
    ):

        raise ValueError(
            f"Non-finite probabilities detected "
            f"for {prefix}."
        )


    y_pred = np.argmax(
        probs,
        axis=1,
    )


    # Explicit labels guarantee a 4 x 4 matrix.
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1,
            2,
            3,
        ],
    )


    # --------------------------------------------------------
    # Export raw confusion-matrix counts
    # --------------------------------------------------------

    raw_cm_file = (
        output_dir
        / f"Raw_CM_{prefix}.csv"
    )

    np.savetxt(
        raw_cm_file,
        cm,
        delimiter=",",
        fmt="%d",
    )


    # --------------------------------------------------------
    # Normalize each row for heat-map intensity
    # while displaying raw sample counts as annotations.
    # --------------------------------------------------------

    row_sums = cm.sum(
        axis=1,
        keepdims=True,
    )

    cm_norm = np.divide(
        cm.astype(float),
        row_sums,
        out=np.zeros_like(
            cm,
            dtype=float,
        ),
        where=row_sums != 0,
    )


    fig, ax = plt.subplots(
        figsize=(5, 5)
    )


    sns.heatmap(
        cm_norm,
        annot=cm,
        fmt="g",
        cmap="Blues",
        cbar=False,
        xticklabels=class_names,
        yticklabels=class_names,
        annot_kws={
            "size": 12,
            "weight": "bold",
        },
        ax=ax,
    )


    ax.set_xlabel(
        "Predicted Phenotype",
        fontweight="bold",
    )

    ax.set_ylabel(
        "True Phenotype",
        fontweight="bold",
    )

    ax.set_title(
        title,
        fontweight="bold",
        pad=15,
    )


    plt.tight_layout()


    figure_file = (
        output_dir
        / f"Fig_CM_{prefix}.pdf"
    )

    plt.savefig(
        figure_file,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


    accuracy = (
        np.trace(cm)
        / np.sum(cm)
        * 100
    )

    print(
        f"  {prefix}: "
        f"accuracy = {accuracy:.2f}%"
    )


# ============================================================
# 7. Generate four one-vs-rest ROC plots
# ============================================================

print(
    "\nGenerating class-specific ROC curves..."
)


for class_index, class_name in enumerate(
    class_names
):

    # Convert the multiclass label into a binary
    # one-vs-rest label for the selected phenotype.
    y_true_binary = (
        y_true == class_index
    ).astype(int)


    fig, ax = plt.subplots(
        figsize=(5, 5)
    )


    for prefix, label, color in zip(
        models_list,
        model_titles,
        colors,
    ):

        y_scores = df_roc[
            f"{prefix}_Prob_{class_name}"
        ].to_numpy()


        fpr, tpr, _ = roc_curve(
            y_true_binary,
            y_scores,
        )

        roc_auc = auc(
            fpr,
            tpr,
        )


        ax.plot(
            fpr,
            tpr,
            color=color,
            linewidth=2,
            label=(
                f"{label} "
                f"(AUC = {roc_auc:.3f})"
            ),
        )


    # Chance-level classifier.
    ax.plot(
        [0, 1],
        [0, 1],
        "k--",
        linewidth=1.5,
    )


    ax.set_xlim(
        [0.0, 1.0]
    )

    ax.set_ylim(
        [0.0, 1.05]
    )


    ax.set_xlabel(
        "False Positive Rate",
        fontweight="bold",
    )

    ax.set_ylabel(
        "True Positive Rate",
        fontweight="bold",
    )

    ax.set_title(
        f"{class_name} Classification",
        fontweight="bold",
        pad=15,
    )


    ax.legend(
        loc="lower right",
        frameon=False,
        fontsize=9,
    )


    sns.despine()

    plt.tight_layout()


    figure_file = (
        output_dir
        / f"Fig_ROC_{class_name}.pdf"
    )

    plt.savefig(
        figure_file,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


    print(
        f"  {class_name} ROC generated."
    )


# ============================================================
# 8. Generate three t-SNE plots
# ============================================================

print(
    "\nGenerating t-SNE plots..."
)


for prefix, title in zip(
    models_list,
    model_titles,
):

    tsne_file = tsne_files[
        prefix
    ]


    df_tsne = pd.read_csv(
        tsne_file
    )


    required_tsne_columns = [
        "t-SNE_1",
        "t-SNE_2",
        "True_Class",
    ]


    missing_tsne_columns = [
        column
        for column in required_tsne_columns
        if column not in df_tsne.columns
    ]


    if missing_tsne_columns:

        raise ValueError(
            f"{tsne_file.name} is missing columns:\n"
            + "\n".join(
                missing_tsne_columns
            )
        )


    if len(df_tsne) != 384:

        raise ValueError(
            f"{tsne_file.name}: "
            f"expected 384 samples, "
            f"but found {len(df_tsne)}."
        )


    fig, ax = plt.subplots(
        figsize=(5.5, 5)
    )


    sns.scatterplot(
        x="t-SNE_1",
        y="t-SNE_2",
        hue="True_Class",
        palette=palette,
        data=df_tsne,
        s=60,
        alpha=0.8,
        edgecolor="w",
        linewidth=0.5,
        ax=ax,
    )


    ax.set_xlabel(
        "t-SNE Dimension 1",
        fontweight="bold",
    )

    ax.set_ylabel(
        "t-SNE Dimension 2",
        fontweight="bold",
    )

    ax.set_title(
        f"Feature Space: {title}",
        fontweight="bold",
        pad=15,
    )


    ax.legend(
        title="Tissue Class",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        frameon=False,
    )


    sns.despine()

    plt.tight_layout()


    figure_file = (
        output_dir
        / f"Fig_tSNE_{prefix}.pdf"
    )

    plt.savefig(
        figure_file,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


    print(
        f"  {prefix} t-SNE generated."
    )


# ============================================================
# 9. Final summary
# ============================================================

print(
    "\n=============================================="
)

print(
    "Figure generation completed successfully."
)

print(
    "=============================================="
)

print(
    "\nGenerated outputs:"
)

print(
    "  3 confusion-matrix PDFs"
)

print(
    "  3 raw confusion-matrix CSV files"
)

print(
    "  4 class-specific ROC PDFs"
)

print(
    "  3 t-SNE PDFs"
)

print(
    f"\nAll newly generated files are located in:\n"
    f"{output_dir}"
)

print(
    "\nThe original Publication_Results directory "
    "was not modified."
)

Repository not found locally. Cloning from GitHub...
Repository root:
/content/3MP-Multimodal-Diagnostic-Platform

Reading publication source data from:
/content/3MP-Multimodal-Diagnostic-Platform/Publication_Results

Generated figures will be saved to:
/content/3MP-Multimodal-Diagnostic-Platform/Generated_Figures

All required publication source files were found.
Loaded 384 out-of-fold predictions.

Generating confusion matrices...
  Bio: accuracy = 60.94%
  Mech: accuracy = 71.61%
  Fused: accuracy = 89.32%

Generating class-specific ROC curves...
  Normal ROC generated.
  Scar ROC generated.
  Inflamed ROC generated.
  Tumor ROC generated.

Generating t-SNE plots...
  Bio t-SNE generated.
  Mech t-SNE generated.
  Fused t-SNE generated.

Figure generation completed successfully.

Generated outputs:
  3 confusion-matrix PDFs
  3 raw confusion-matrix CSV files
  4 class-specific ROC PDFs
  3 t-SNE PDFs

All newly generated files are located in:
/content/3MP-Multimodal-Diagnostic-Platf